In [1]:
# Library import

import pandas as pd
import numpy as np
from pathlib import Path
import os

# Dataset folder
DATA_DIR = Path("../data")

files = sorted(DATA_DIR.glob("*.csv"))
print("CSV files found:")
for f in files:
    print(f"--> {f.name}")   

    

CSV files found:
--> gt_2011.csv
--> gt_2012.csv
--> gt_2013.csv
--> gt_2014.csv
--> gt_2015.csv


In [2]:
# Load all yearly datasets

dfs = []

for f in files:
    df = pd.read_csv(f)
    year = f.stem.split("_")[-1]
    df["Year"] = int(year)
    dfs.append(df)
data = pd.concat(dfs,ignore_index=True)  

print("\nDataset loaded successfully")
print(f"Shape is {data.shape}")


Dataset loaded successfully
Shape is (36733, 12)


In [3]:
#Basic dataset information

print("\n----------Dataset Shape----------")
print(f"Rows --> {data.shape[0]}")
print(f"Columns --> {data.shape[1]}")

print("\n----------Column Names----------")
print(data.columns.tolist())

print("\n----------Data Types----------")
print(data.dtypes)

print("\n----------First 5 Rows----------")
display(data.head())

print("\n----------Last 5 Rows----------")
display(data.tail())


----------Dataset Shape----------
Rows --> 36733
Columns --> 12

----------Column Names----------
['AT', 'AP', 'AH', 'AFDP', 'GTEP', 'TIT', 'TAT', 'TEY', 'CDP', 'CO', 'NOX', 'Year']

----------Data Types----------
AT      float64
AP      float64
AH      float64
AFDP    float64
GTEP    float64
TIT     float64
TAT     float64
TEY     float64
CDP     float64
CO      float64
NOX     float64
Year      int64
dtype: object

----------First 5 Rows----------


,AT,AP,AH,AFDP,GTEP,TIT,TAT,TEY,CDP,CO,NOX,Year
0,4.5878,1018.7,83.675,3.5758,23.979,1086.2,549.83,134.67,11.898,0.32663,81.952,2011
1,4.2932,1018.3,84.235,3.5709,23.951,1086.1,550.05,134.67,11.892,0.44784,82.377,2011
2,3.9045,1018.4,84.858,3.5828,23.990,1086.5,550.19,135.10,12.042,0.45144,83.776,2011
3,3.7436,1018.3,85.434,3.5808,23.911,1086.5,550.17,135.03,11.990,0.23107,82.505,2011
4,3.7516,1017.8,85.182,3.5781,23.917,1085.9,550.00,134.67,11.910,0.26747,82.028,2011



----------Last 5 Rows----------


,AT,AP,AH,AFDP,GTEP,TIT,TAT,TEY,CDP,CO,NOX,Year
36728,3.6268,1028.5,93.200,3.1661,19.087,1037.0,541.59,109.08,10.411,10.9930,89.172,2015
36729,4.1674,1028.6,94.036,3.1923,19.016,1037.6,542.28,108.79,10.344,11.1440,88.849,2015
36730,5.4820,1028.5,95.219,3.3128,18.857,1038.0,543.48,107.81,10.462,11.4140,96.147,2015
36731,5.8837,1028.7,94.200,3.9831,23.563,1076.9,550.11,131.41,11.771,3.3134,64.738,2015
36732,6.0392,1028.8,94.547,3.8752,22.524,1067.9,548.23,125.41,11.462,11.9810,109.240,2015


In [4]:
# ------------------------------------------------------------
#Number of observations in each year
# ------------------------------------------------------------

year_counts = data["Year"].value_counts().sort_index()

print("Observations by year:")
display(year_counts.to_frame("Number of Observations"))

Observations by year:


,Number of Observations
Year,
2011,7411
2012,7628
2013,7152
2014,7158
2015,7384


In [5]:
# ------------------------------------------------------------
#Missing values
# ------------------------------------------------------------

missing = pd.DataFrame({
    "Missing_Count": data.isna().sum(),
    "Missing_Percentage": data.isna().mean() * 100
})

print("========== MISSING VALUE AUDIT ==========")
display(missing)

========== MISSING VALUE AUDIT ==========


,Missing_Count,Missing_Percentage
AT,0,0.0
AP,0,0.0
AH,0,0.0
AFDP,0,0.0
GTEP,0,0.0
TIT,0,0.0
TAT,0,0.0
TEY,0,0.0
CDP,0,0.0
CO,0,0.0


In [6]:
# ------------------------------------------------------------
#Infinite values
# ------------------------------------------------------------

numeric_cols = data.select_dtypes(include=np.number).columns

infinite_counts = np.isinf(data[numeric_cols]).sum()

print("========== INFINITE VALUE AUDIT ==========")
display(infinite_counts.to_frame("Infinite_Count"))

========== INFINITE VALUE AUDIT ==========


,Infinite_Count
AT,0
AP,0
AH,0
AFDP,0
GTEP,0
TIT,0
TAT,0
TEY,0
CDP,0
CO,0


In [7]:
# ------------------------------------------------------------
#Duplicate-row audit
# ------------------------------------------------------------

total_duplicates = data.duplicated().sum()

print("========== DUPLICATE AUDIT ==========")
print("Total duplicate rows:", total_duplicates)

duplicates_by_year = data.groupby("Year").apply(
    lambda x: x.duplicated().sum()
)

print("\nDuplicates by year:")
display(duplicates_by_year.to_frame("Duplicate_Rows"))

========== DUPLICATE AUDIT ==========
Total duplicate rows: 7

Duplicates by year:


,Duplicate_Rows
Year,
2011,0
2012,0
2013,0
2014,7
2015,0


In [8]:
# ------------------------------------------------------------
#Descriptive statistics
# ------------------------------------------------------------

print("========== DESCRIPTIVE STATISTICS ==========")

summary = data[numeric_cols].describe().T

summary["missing"] = data[numeric_cols].isna().sum()
summary["unique"] = data[numeric_cols].nunique()

display(summary)

========== DESCRIPTIVE STATISTICS ==========


,count,mean,std,min,25%,50%,75%,max,missing,unique
AT,36733.0,17.712726,7.447451,-6.234800,11.7810,17.8010,23.6650,37.1030,0,22523
AP,36733.0,1013.070165,6.463346,985.850000,1008.8000,1012.6000,1017.0000,1036.6000,0,791
AH,36733.0,77.867015,14.461355,24.085000,68.1880,80.4700,89.3760,100.2000,0,25708
AFDP,36733.0,3.925518,0.773936,2.087400,3.3556,3.9377,4.3769,7.6106,0,20495
GTEP,36733.0,25.563801,4.195957,17.698000,23.1290,25.1040,29.0610,40.7160,0,12967
TIT,36733.0,1081.428084,17.536373,1000.800000,1071.8000,1085.9000,1097.0000,1100.9000,0,799
TAT,36733.0,546.158517,6.842360,511.040000,544.7200,549.8800,550.0400,550.6100,0,2769
TEY,36733.0,133.506404,15.618634,100.020000,124.4500,133.7300,144.0800,179.5000,0,6236
CDP,36733.0,12.060525,1.088795,9.851800,11.4350,11.9650,12.8550,15.1590,0,4447
CO,36733.0,2.372468,2.262672,0.000388,1.1824,1.7135,2.8429,44.1030,0,26185


In [9]:
# ------------------------------------------------------------
#Physical range / extreme-value audit
# ------------------------------------------------------------

range_table = pd.DataFrame({
    "Minimum": data[numeric_cols].min(),
    "Maximum": data[numeric_cols].max(),
    "Mean": data[numeric_cols].mean(),
    "Std": data[numeric_cols].std(),
    "Median": data[numeric_cols].median()
})

display(range_table)

,Minimum,Maximum,Mean,Std,Median
AT,-6.234800,37.1030,17.712726,7.447451,17.8010
AP,985.850000,1036.6000,1013.070165,6.463346,1012.6000
AH,24.085000,100.2000,77.867015,14.461355,80.4700
AFDP,2.087400,7.6106,3.925518,0.773936,3.9377
GTEP,17.698000,40.7160,25.563801,4.195957,25.1040
TIT,1000.800000,1100.9000,1081.428084,17.536373,1085.9000
TAT,511.040000,550.6100,546.158517,6.842360,549.8800
TEY,100.020000,179.5000,133.506404,15.618634,133.7300
CDP,9.851800,15.1590,12.060525,1.088795,11.9650
CO,0.000388,44.1030,2.372468,2.262672,1.7135


In [10]:
# ------------------------------------------------------------
#Year-wise statistics
# ------------------------------------------------------------

year_stats = data.groupby("Year")[numeric_cols].agg(
    ["mean", "std", "min", "max"]
)

display(year_stats)

AT                                      AP                    \
           mean       std      min     max         mean       std     min   
Year                                                                        
2011  17.112261  7.428307  2.11630  34.532  1014.167883  6.293003  995.79   
2012  18.397950  7.661038  0.52230  34.929  1012.258153  6.384291  985.85   
2013  17.602620  6.862890  0.28985  33.873  1011.999607  6.290065  989.38   
2014  18.217071  6.994920  1.14440  34.189  1012.384261  5.973395  994.84   
2015  17.225259  8.095783 -6.23480  37.103  1014.509110  6.895430  989.40   

                     AH             ...        CO                NOX  \
         max       mean        std  ...       min     max       mean   
Year                                ...                                
2011  1034.2  79.174989  13.465898  ...  0.000388  43.622  67.575392   
2012  1031.8  79.074804  14.105087  ...  0.007505  44.103  68.788965   
2013  1029.7  80.461624  14.125390  ...  0.005033  35.045  70.007899   
2014  1031.4  82.143926  13.000424  ...  0.001640  34.820  60.066886   
2015  1036.6  68.647464  13.541116  ...  0.212800  41.097  59.890509   

                                   Year                   
            std     min     max    mean  std   min   max  
Year                                                      
2011  10.683331  27.765  119.32  2011.0  0.0  2011  2011  
2012  10.224937  41.777  119.89  2012.0  0.0  2012  2012  
2013  12.048249  43.198  119.91  2013.0  0.0  2013  2013  
2014   9.971729  27.183  118.47  2014.0  0.0  2014  2014  
2015  11.132464  25.905  119.68  2015.0  0.0  2015  2015  

[5 rows x 48 columns]

In [11]:
# Mean of each variable by year

year_means = data.groupby("Year")[numeric_cols].mean()

display(year_means)

,AT,AP,AH,AFDP,GTEP,TIT,TAT,TEY,CDP,CO,NOX,Year
Year,,,,,,,,,,,,
2011,17.112261,1014.167883,79.174989,4.090755,25.663721,1084.733909,544.503170,135.745675,12.207578,1.572486,67.575392,2011.0
2012,18.397950,1012.258153,79.074804,4.306717,25.181361,1082.890233,546.263793,132.675552,12.000121,2.361133,68.788965,2012.0
2013,17.602620,1011.999607,80.461624,3.695958,25.105097,1081.569463,545.780885,132.168342,11.971586,2.723031,70.007899,2013.0
2014,18.217071,1012.384261,82.143926,3.914499,25.741993,1078.836854,547.638254,132.907978,12.023857,2.081103,60.066886,2014.0
2015,17.225259,1014.509110,68.647464,3.598909,26.130149,1078.974689,546.642484,133.993380,12.097025,3.129986,59.890509,2015.0


In [12]:
# ------------------------------------------------------------
#Constant / near-constant columns
# ------------------------------------------------------------

constant_cols = [
    col for col in numeric_cols
    if data[col].nunique() <= 1
]

print("Constant columns:", constant_cols)

for col in numeric_cols:
    print(f"{col:10s} -> {data[col].nunique():6,d} unique values")

Constant columns: []
AT         -> 22,523 unique values
AP         ->    791 unique values
AH         -> 25,708 unique values
AFDP       -> 20,495 unique values
GTEP       -> 12,967 unique values
TIT        ->    799 unique values
TAT        ->  2,769 unique values
TEY        ->  6,236 unique values
CDP        ->  4,447 unique values
CO         -> 26,185 unique values
NOX        -> 23,637 unique values
Year       ->      5 unique values


In [13]:
# ------------------------------------------------------------
#Detailed duplicate audit
# ------------------------------------------------------------

for year in sorted(data["Year"].unique()):
    
    year_df = data[data["Year"] == year]
    dup = year_df[year_df.duplicated(keep=False)]
    
    print(f"\nYear {year}")
    print("-" * 40)
    print("Rows:", len(year_df))
    print("Duplicate rows:", len(dup))
    
    if len(dup) > 0:
        display(dup.head(20))


Year 2011
----------------------------------------
Rows: 7411
Duplicate rows: 0

Year 2012
----------------------------------------
Rows: 7628
Duplicate rows: 0

Year 2013
----------------------------------------
Rows: 7152
Duplicate rows: 0

Year 2014
----------------------------------------
Rows: 7158
Duplicate rows: 9


,AT,AP,AH,AFDP,GTEP,TIT,TAT,TEY,CDP,CO,NOX,Year
26398,23.156,1004.2,95.938,4.0547,24.672,1076.6,549.87,127.01,11.835,1.7056,47.352,2014
26399,23.156,1004.2,95.938,4.0547,24.672,1076.6,549.87,127.01,11.835,1.7056,47.352,2014
26400,23.156,1004.2,95.938,4.0547,24.672,1076.6,549.87,127.01,11.835,1.7056,47.352,2014
26401,23.156,1004.2,95.938,4.0547,24.672,1076.6,549.87,127.01,11.835,1.7056,47.352,2014
26402,23.156,1004.2,95.938,4.0547,24.672,1076.6,549.87,127.01,11.835,1.7056,47.352,2014
27388,26.067,1008.3,87.328,5.0703,29.984,1099.1,546.78,146.14,13.038,1.3163,52.564,2014
27389,26.067,1008.3,87.328,5.0703,29.984,1099.1,546.78,146.14,13.038,1.3163,52.564,2014
27390,26.067,1008.3,87.328,5.0703,29.984,1099.1,546.78,146.14,13.038,1.3163,52.564,2014
27391,26.067,1008.3,87.328,5.0703,29.984,1099.1,546.78,146.14,13.038,1.3163,52.564,2014



Year 2015
----------------------------------------
Rows: 7384
Duplicate rows: 0


In [14]:
# ------------------------------------------------------------
#Cross-year duplicate audit
# ------------------------------------------------------------

feature_cols = [c for c in data.columns if c != "Year"]

duplicate_features = data[
    data.duplicated(subset=feature_cols, keep=False)
].sort_values(feature_cols)

print("Rows having identical measurement values across records:")
print(len(duplicate_features))

display(duplicate_features.head(20))

Rows having identical measurement values across records:
9


,AT,AP,AH,AFDP,GTEP,TIT,TAT,TEY,CDP,CO,NOX,Year
26398,23.156,1004.2,95.938,4.0547,24.672,1076.6,549.87,127.01,11.835,1.7056,47.352,2014
26399,23.156,1004.2,95.938,4.0547,24.672,1076.6,549.87,127.01,11.835,1.7056,47.352,2014
26400,23.156,1004.2,95.938,4.0547,24.672,1076.6,549.87,127.01,11.835,1.7056,47.352,2014
26401,23.156,1004.2,95.938,4.0547,24.672,1076.6,549.87,127.01,11.835,1.7056,47.352,2014
26402,23.156,1004.2,95.938,4.0547,24.672,1076.6,549.87,127.01,11.835,1.7056,47.352,2014
27388,26.067,1008.3,87.328,5.0703,29.984,1099.1,546.78,146.14,13.038,1.3163,52.564,2014
27389,26.067,1008.3,87.328,5.0703,29.984,1099.1,546.78,146.14,13.038,1.3163,52.564,2014
27390,26.067,1008.3,87.328,5.0703,29.984,1099.1,546.78,146.14,13.038,1.3163,52.564,2014
27391,26.067,1008.3,87.328,5.0703,29.984,1099.1,546.78,146.14,13.038,1.3163,52.564,2014


In [15]:
# ============================================================
# FINAL DATA AUDIT REPORT
# ============================================================

print("=" * 65)
print("                 DATA AUDIT REPORT")
print("=" * 65)

print(f"Total observations       : {len(data):,}")
print(f"Total variables          : {len(feature_cols)}")
print(f"Years                    : {sorted(data['Year'].unique())}")
print(f"Missing values           : {data[feature_cols].isna().sum().sum():,}")
print(f"Infinite values          : {np.isinf(data[feature_cols]).sum().sum():,}")
print(f"Duplicate complete rows  : {data.duplicated().sum():,}")
print(f"Constant columns         : {len(constant_cols)}")

print("\nObservations by year:")
for year, count in data["Year"].value_counts().sort_index().items():
    print(f"  {year}: {count:,}")

print("\n" + "=" * 65)
print("AUDIT COMPLETE")
print("=" * 65)

                 DATA AUDIT REPORT
Total observations       : 36,733
Total variables          : 11
Years                    : [np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015)]
Missing values           : 0
Infinite values          : 0
Duplicate complete rows  : 7
Constant columns         : 0

Observations by year:
  2011: 7,411
  2012: 7,628
  2013: 7,152
  2014: 7,158
  2015: 7,384

AUDIT COMPLETE
